## The corpus

The corpus holds several Palestinian Targum manuscripts: Targum Pseudo-Jonathan, the Fragment
Targums, Cairo Genizah fragments, and Targum Neofiti. Neofiti is represented in two views, a
'base' text that keeps only unmarked segments (no variants, no interlinear or in-line additions),
and a 'full' text that places every segment side by side. The text is Jewish Palestinian Aramaic,
carried in ETCBC transliteration and in Aramaic square script.

The object hierarchy is `text > book > chapter > verse > segment > word_group > word`, and `word`
is the slot type. `word_group` is included as a step toward full consistency across the hierarchy
while the underlying data model continues to develop. The segment `type` feature records the
editorial state of a run of text, with values including `unmarked_text`, `variant`, `addition`,
`editorial_deletion`, `broken_text`, and `interlinear`. Segments and words both carry `start_idx`
and `end_idx`, one-based character offsets into the `line` feature of the verse that contains
them. A segment's offsets are the minimum and the maximum of the offsets of the words it holds, so
the two tiers agree by construction.

Two features record how a word relates to the published line. `source` is `word` where
morphological analysis was placed directly on the word, `word_group` where the group carries a
surface form that the analysis does not individually cover, `none` where the manuscript records a
word that carries no morphological analysis at all, and `segment` where the word stands in for a
segment that no analysed word reaches, carrying that segment's own text so the segment can still
be published. `part` counts the pieces of a word that an editorial boundary cuts in two. The two
pieces share their `word_group` and its `id`, each sits in the segment that its own characters
fall in, and a word with `part` 1 and no sibling is whole. Because a split word group occupies two
nodes, the tests below that speak of a word group treat all nodes sharing one `id` as a single
unit.

## Baseline tests

Tests 1 to 5 move up through the object hierarchy and ask whether a given tier of nodes
reconstructs the tier above it, for example whether the segments of a verse reconstruct the verse
line. Test 4 also resolves the segment offsets against the line they index. Test 6 checks whether
the stored ordinals agree with the order in which the dataset publishes its nodes.

Each test reports its figures per corpus, because the five corpora differ substantially in how
much morphological analysis their source material provides.

### Reading the results

Tests 1, 3, 4, and 5 report figures that vary by corpus, reflecting properties of the source
material. Most of the failures fall into two categories, which the second table of test 1
isolates: surface forms whose morphological analysis covers only part of the word, and a diacritic
written one way at the word tier and another way at the word group tier.

Test 2 reports a figure that describes the publication boundary itself. A word that straddles two
editorial states is contained by only one segment, leaving the other segment without a word group
that spells it. The same effect limits the proportion of segment spans whose slice returns the
segment, and the proportion of verses whose segments spell the line, both reported in test 4.

In [ ]:
# Loads the dataset and reports its provenance. The version string, the version of the XML the
# dataset was compiled from, and the compilation timestamp are read from the metadata of the
# `otype` feature, which every Text-Fabric dataset carries. The node counts printed here are the
# quantities every later test is measured against.

import os
from pathlib import Path

from dotenv import load_dotenv
from tf.app import use

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/xml.yml").exists())
load_dotenv(REPO_ROOT / ".env")
ORG = 'etcbc' if not os.environ["TF_ORG"] else os.environ["TF_ORG"]
REPO = 'targum' if not os.environ["TF_RPO"] else os.environ["TF_REPO"]

A = use(f"{ORG}/{REPO}:clone", checkout="clone", version="0.0.1-alpha", hoist=globals())

meta = A.api.TF.features["otype"].metaData

In [ ]:
# Checks the dataset's provenance against the files it was compiled from. The dataset is stale
# unless both checks below hold.
#
#   1  The xmlVersion metadata against the newest directory under `xml/`. A dataset compiled from
#      an older XML directory than the one currently present is behind by at least that much.
#   2  The feature descriptions against `config/xml.yml`. A mismatch here means the dataset was
#      compiled before the current feature documentation was written.
#
# A failure in either check means the figures below describe an outdated build of the dataset.

import re
import yaml

FRESH = True


def stale(what, got, want):
    global FRESH
    FRESH = False
    print(f"  STALE  {what}\n           dataset: {got}\n           source:  {want}")


xml_versions = sorted(p.name for p in (REPO_ROOT / "xml").iterdir() if p.is_dir())
newest_xml = xml_versions[-1] if xml_versions else None
print(f"xmlVersion {meta.get('xmlVersion')}  written {meta.get('dateWritten')}")
print(f"newest xml/ {newest_xml}  (of {len(xml_versions)})")
if newest_xml and meta.get("xmlVersion") != newest_xml:
    stale("the dataset was compiled from an older XML than xml/ holds",
          meta.get("xmlVersion"), newest_xml)

declared = yaml.safe_load((REPO_ROOT / "config/xml.yml").read_text())["featureDescriptions"]


def norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip()


for name, want in declared.items():
    feat = A.api.TF.features.get(name)
    if feat is None:
        continue
    got = feat.metaData.get("description")
    if norm(got) != norm(want):
        stale(f"feature {name!r} carries a description config/xml.yml no longer declares",
              norm(got)[:110], norm(want)[:110])

counts = {t: n for t, n in ((t, len(list(F.otype.s(t)))) for t in F.otype.all)}
print("nodes:", ", ".join(f"{t} {counts[t]:,}" for t in F.otype.all))

print()
print("dataset is current" if FRESH else "DATASET IS STALE, rebuild before reading on")
assert FRESH, "stale dataset: see the report above"

In [ ]:
# Reports the feature inventory and the distribution of the three features that the tests below
# depend on. `source` determines which words carry a morphological analysis, `reading` identifies
# the readings that compete for one position, and the segment `type` feature records the editorial
# state of a run of the line.

FEATURES = sorted(f for f in A.api.TF.features if not f.startswith("o") and f != "otext")
for otype in F.otype.all:
    sample = F.otype.s(otype)[:400]
    present = sorted(f for f in FEATURES
                     if hasattr(A.api.F, f) and any(A.api.Fs(f).v(n) is not None for n in sample))
    print(f"{otype:<11} {', '.join(present)}")
print()
print("source     ", dict(F.source.freqList()))
print("reading    ", dict(F.reading.freqList()))
print("segment    ", dict(F.type.freqList()))
print()
for fmt in sorted(T.formats):
    print(f"  {fmt:<16} {T.text(range(1, 9), fmt=fmt)}")

In [ ]:
# Defines the material shared by every test below.
#
# `letters` is the comparison class used throughout. It strips whitespace so that strings drawn
# from different tiers of the corpus, which do not always agree on how much space they record, can
# still be compared fairly.
#
# `verse_nodes` keys each verse by the pair (manuscript, uid). The `uid` feature is unique within
# one corpus and repeats across corpora, so the manuscript name is needed to make the key unique.
# Two verses of the Genizah material occupy more than one verse node, because the order in which
# the dataset publishes their words does not keep the rows of those verses contiguous. The tests
# below aggregate over the list of nodes that share a key.
#
# `table` prints one row per corpus with a total, and takes derived percentage columns as
# (label, numerator, denominator) triples.

import re
from collections import Counter, defaultdict

_WS = re.compile(r"\s+")


def letters(s):
    return _WS.sub("", s or "")


CORPORA = ["cairo_genizah", "frag_targ", "neofiti_base_raw", "neofiti_full", "psj"]
_BY_NAME = {"neofiti_base": "neofiti_base_raw", "neofiti_full": "neofiti_full",
            "pseudo_jonathan": "psj"}


def corpus_of(text_node):
    name = F.text.v(text_node)
    if name.startswith("cairo"):
        return "cairo_genizah"
    if name.startswith("fragment_targum"):
        return "frag_targ"
    return _BY_NAME[name]


verse_nodes = defaultdict(list)
corpus_of_verse = {}
for _t in F.otype.s("text"):
    _name, _corpus = F.text.v(_t), corpus_of(_t)
    for _v in L.d(_t, otype="verse"):
        verse_nodes[(_name, F.uid.v(_v))].append(_v)
        corpus_of_verse[(_name, F.uid.v(_v))] = _corpus


def table(title, rows, cols, pct=(), key="corpus"):
    keys = [k for k in CORPORA if k in rows] + [k for k in rows if k not in CORPORA]
    totals = Counter()
    for k in keys:
        totals.update(rows[k])
    head = [key, *cols, *(lbl for lbl, _, _ in pct)]
    body = []
    for k in [*keys, "TOTAL"]:
        r = totals if k == "TOTAL" else rows[k]
        body.append([k, *(f"{r[c]:,}" for c in cols),
                     *(f"{100.0 * r[n] / r[d]:.2f}" if r[d] else "-" for _, n, d in pct)])
    w = [max(len(str(x)) for x in col) for col in zip(head, *body)]
    rule = "  ".join("-" * i for i in w)
    print(title)
    print("  " + "  ".join(h.rjust(i) for h, i in zip(head, w)))
    print("  " + rule)
    for line in body[:-1]:
        print("  " + "  ".join(str(x).rjust(i) for x, i in zip(line, w)))
    print("  " + rule)
    print("  " + "  ".join(str(x).rjust(i) for x, i in zip(body[-1], w)))


results = {}
print(f"{len(verse_nodes):,} verses distributed over "
      f"{sum(len(v) for v in verse_nodes.values()):,} verse nodes")

In [ ]:
# TEST 1. Composition of a word group from its words.
#
# Method: for every word group, concatenate the `voc` feature of the `word` nodes it contains, in
# slot order, and compare the result to the group's own `g_voc` feature under the letters
# comparison.
#
# A word group is treated here by its `id`, not as a single node. Where an editorial boundary cuts
# a word in two, each piece sits in a different segment, so the dataset publishes a second
# `word_group` node for it. Both nodes carry the whole source word as `g_voc`, and each holds only
# its own words. Reading such a group node by node would report it as failing twice for no real
# reason. Reading it by `id`, which is what this test does, states the intended claim: the
# morphological analysis of a word accounts for that word.
#
# What it measures: whether the morphological decomposition accounts for the surface form it was
# derived from. A group whose words spell it exactly is one where every character of the
# manuscript word is claimed by some morpheme.
#
# Interpretation: `analysed` restricts the population to the groups whose words all carry
# morphological analysis, the set the decomposition can be evaluated against. The second table
# classifies the failures. `reordered` counts groups where the same letters appear in a different
# order, an alignment fault. `words_shorter` counts groups where the analysis covers only part of
# the surface form, a property of the source material. `dot_vs_mark` counts groups where the two
# strings differ only in whether a Genizah diacritic is written as a full stop or as the combining
# mark U+0740.

test1 = defaultdict(Counter)
group_failures = []
for _k, _vns in verse_nodes.items():
    _c = corpus_of_verse[_k]
    # The pieces of one word group, in node order, keyed by the group's own id.
    _by_id = defaultdict(list)
    for _v in _vns:
        for _g in L.d(_v, otype="word_group"):
            _by_id[F.id.v(_g)].append(_g)
    for _gid, _nodes in _by_id.items():
        ws = [w for _g in _nodes for w in L.d(_g, otype="word")]
        built = "".join(F.voc.v(w) or "" for w in ws)
        target = F.g_voc.v(_nodes[0]) or ""
        analysed = {F.source.v(w) for w in ws} == {"word"}
        ok = letters(built) == letters(target)
        _r = test1[_c]
        _r["groups"] += 1
        _r["words"] += len(ws)
        _r["split_groups"] += len(_nodes) > 1
        _r["words_spell_group"] += ok
        _r["differ"] += not ok
        _r["analysed"] += analysed
        _r["analysed_spelled"] += analysed and ok
        if not ok:
            group_failures.append((_c, _nodes[0], target, built))
            _b, _t = letters(built), letters(target)
            _r["reordered"] += sorted(_b) == sorted(_t)
            _r["words_shorter"] += len(_b) < len(_t)
            _r["words_longer"] += len(_b) > len(_t)
            _r["dot_vs_mark"] += _b.replace("݀", ".") == _t

table("TEST 1  the words of a word group spell the group", test1,
      ["groups", "split_groups", "words", "words_spell_group", "differ", "analysed",
       "analysed_spelled"],
      pct=[("%", "words_spell_group", "groups"), ("% of analysed", "analysed_spelled", "analysed")])
print()
table("TEST 1  the composition of the failures", test1,
      ["differ", "reordered", "words_shorter", "words_longer", "dot_vs_mark"])
results["1 words compose the word group"] = (sum(test1[c]["groups"] for c in CORPORA),
                                             sum(test1[c]["words_spell_group"] for c in CORPORA))

In [ ]:
# Displays four of the word groups that test 1 reports as failing, followed by one of them
# rendered by the Text-Fabric app. The printed line gives the published surface form, the form the
# words spell, and the analysis of each word, so the disagreement can be inspected directly. The
# second table of test 1 already summarises the failures as a population.

for _c, _g, _target, _built in group_failures[:4]:
    print(f"{_c:<16} word group {_g}: published {_target!r}, its words spell {_built!r}")
    print(f"{'':<16}   words: "
          + str([(F.voc.v(w), F.lex.v(w), F.sp.v(w), F.source.v(w)) for w in L.d(_g, otype='word')]))

_c, _g, _target, _built = next((f for f in group_failures if f[0] == "neofiti_full"),
                               group_failures[0])
print(f"\n{_c}: word group {_g}")
A.pretty(_g, withNodes=True)

In [ ]:
# TEST 2. Composition of a segment from the words it contains.
#
# Method: for every `segment` node, concatenate the `voc` feature of the `word` nodes it contains
# and compare the result to the segment's own `segment` feature under the letters comparison.
#
# The word tier is the correct level to measure at here. `g_voc` records the whole source word, so
# a word cut by an editorial boundary contributes only its own characters to each side, and a
# segment holding half a word would be compared against the whole word and fail by construction.
# The word tier carries exactly the characters of the segment it sits in. `whole_groups` keeps the
# group-level comparison for reference, and it fails wherever a word straddles two editorial
# states.
#
# `all_readings` uses every word in the segment. `base_reading` uses only the words of groups
# whose `reading` is 0, the underlying reading the first scribe wrote. A segment holding no
# variant satisfies both identically.
#
# What it measures: whether the word tier accounts for the editorial segment that contains it.
# Each word is attributed to the segment its own span falls in, and a word whose span crosses a
# segment boundary is split so that each piece belongs to its own segment.

test2 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _c = corpus_of_verse[_k]
    for _v in _vns:
        for _s in L.d(_v, otype="segment"):
            gs = L.d(_s, otype="word_group")
            ws = L.d(_s, otype="word")
            target = letters(F.segment.v(_s))
            every = letters("".join(F.voc.v(w) or "" for w in ws))
            base = letters("".join(F.voc.v(w) or "" for g in gs if F.reading.v(g) == 0
                                   for w in L.d(g, otype="word")))
            whole = letters("".join(F.g_voc.v(g) or "" for g in gs))
            _r = test2[_c]
            _r["segments"] += 1
            _r["with_words"] += bool(ws)
            _r["no_letters"] += not target
            _r["has_variants"] += any(F.reading.v(g) != 0 for g in gs)
            _r["filler_only"] += bool(ws) and all(F.source.v(w) == "segment" for w in ws)
            _r["all_readings"] += every == target
            _r["base_reading"] += base == target
            _r["either"] += (every == target or base == target)
            _r["whole_groups"] += whole == target

table("TEST 2  the words of a segment spell the segment", test2,
      ["segments", "with_words", "no_letters", "has_variants", "filler_only",
       "all_readings", "base_reading", "either", "whole_groups"],
      pct=[("% either", "either", "segments"),
           ("% whole groups", "whole_groups", "segments")])
results["2 words compose the segment"] = (sum(test2[c]["segments"] for c in CORPORA),
                                          sum(test2[c]["either"] for c in CORPORA))

In [ ]:
# TEST 3. Reconstruction of the verse line from the word group tier.
#
# Method: concatenate the `g_voc` feature of every word group of a verse, in publication order,
# and compare the result to the verse's `line` feature under the letters comparison. Variant
# readings are included, because the published line carries them inline.
#
# Each word group is counted once, by its `id`. A group that an editorial boundary cuts in two
# publishes as two nodes carrying the same `g_voc`, so concatenating over nodes instead of ids
# would double-count that word's surface form and fail the verse for an unrelated reason.
#
# What it measures: whether the surface forms recorded at the word group tier account for the
# verse line the dataset publishes. This is the strongest statement available at that tier, and it
# is independent of the morphological decomposition tested in test 1.

test3 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = test3[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    _seen, _built = set(), []
    for _v in _vns:
        for _g in L.d(_v, otype="word_group"):
            _gid = F.id.v(_g)
            if _gid in _seen:
                continue
            _seen.add(_gid)
            _built.append(F.g_voc.v(_g) or "")
    _r["verses"] += 1
    _r["word_groups"] += len(_seen)
    _r["groups_spell_line"] += letters("".join(_built)) == letters(line)

table("TEST 3  the word groups of a verse spell the verse line", test3,
      ["verses", "word_groups", "groups_spell_line"],
      pct=[("%", "groups_spell_line", "verses")])
results["3 word groups compose the verse line"] = (sum(test3[c]["verses"] for c in CORPORA),
                                                   sum(test3[c]["groups_spell_line"] for c in CORPORA))

In [ ]:
# TEST 4. Reconstruction of the verse line from the segment tier.
#
# Method: concatenate the `segment` feature of every `segment` node of a verse, in publication
# order, and compare the result to the verse's `line` feature under the letters comparison.
# Variant segments are included, because the published line carries the competing readings
# inline. Restricting the concatenation to segments that are not variants would still recover no
# verse that the full concatenation loses. `utf8_spell_line` repeats the comparison in Aramaic
# square script. `repeated` counts the segments a verse publishes more than once under one `id`,
# so the concatenation emits their text twice.
#
# The same loop resolves the segment offsets against the line, which the cell below reports. The
# line cut at a segment's span is compared to the segment's own string, and the spans of a verse
# are then examined as a partition. They must not overlap, the material between two consecutive
# spans must be whitespace only, and the material before the first span and after the last span
# must also be whitespace. A verse that satisfies all three conditions has segments that tile the
# line. Whether the spans also ascend in publication order is test 6's question, asked of every
# tier at once. The word tier carries its own `start_idx` and `end_idx`, resolved two cells below.
# A segment's span is the minimum and the maximum of the spans of the words it holds, so the two
# tiers agree by construction, and the word-level statement is the stronger of the two.
#
# What it measures: whether the editorial tier accounts for the verse line the dataset publishes.
# Test 3 asks that question of the word group tier and test 5 asks it of the word tier. The
# segment is the tier between them, and the only one of the three whose strings are runs of the
# line itself, independent of morphology.
#
# Interpretation: the second table restricts the population by the morphological analysis
# underneath. A segment carries morphological analysis when every word it holds has `source`
# `word`. A verse is analysed when every one of its segments does. That is the population the
# decomposition can be evaluated against, and it is reported beside the overall figure and beside
# the rate for the verses that hold an unanalysed segment, so the three can be read separately, as
# tests 1 and 5 do at their own tiers. The third table attributes the failures.
# `text_differs_from_its_span` counts the verses holding a segment whose stored string does not
# match the slice of the line named by its own offsets, the same containment boundary that test 2
# measures from the other side. `spans_still_cover_the_line` counts the failures whose geometry
# still returns the whole line, meaning the loss originates in the segment strings while the spans
# remain correct. The fourth table gives the proportion of the letters of each line that the
# segments emit at all. This proportion is high everywhere, so the shortfall consists of many
# small local losses spread across the text.

test4 = defaultdict(Counter)
segment_failures = []
for _k, _vns in verse_nodes.items():
    _r = test4[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    line_utf8 = max({F.line_utf8.v(v) or "" for v in _vns})
    segs = [s for v in _vns for s in L.d(v, otype="segment")]
    target = letters(line)
    built = letters("".join(F.segment.v(s) or "" for s in segs))
    ok = built == target
    analysed, spans = [], set()
    for s in segs:
        ws = L.d(s, otype="word")
        analysed.append(bool(ws) and all(F.source.v(w) == "word" for w in ws))
        a, b = F.start_idx.v(s), F.end_idx.v(s)
        if a is None:
            _r["segment_no_offset"] += 1
            continue
        _r["segment_offset"] += 1
        _r["segment_slice_is_segment"] += line[a - 1:b] == (F.segment.v(s) or "")
        spans.add((a, b))
    every_analysed = bool(analysed) and all(analysed)
    _r["verses"] += 1
    _r["segments"] += len(segs)
    _r["repeated"] += len(segs) - len({F.id.v(s) for s in segs})
    _r["segments_spell_line"] += ok
    _r["utf8_spell_line"] += letters(
        "".join(F.segment_utf8.v(s) or "" for s in segs)) == letters(line_utf8)
    _r["line_letters"] += len(target)
    _r["segment_letters"] += len(built)
    _r["with_morphology"] += sum(analysed)
    _r["without_morphology"] += len(analysed) - sum(analysed)
    _r["analysed_verses"] += every_analysed
    _r["analysed_spell_line"] += every_analysed and ok
    _r["rest_verses"] += not every_analysed
    _r["rest_spell_line"] += (not every_analysed) and ok
    if spans:
        # A verse whose rows are not contiguous under the publication order is emitted as more
        # than one verse node, and a segment spanning the break is emitted once per node. The
        # partition is a statement about distinct spans, so they are collected as a set.
        ordered = sorted(spans)
        disjoint = all(nxt[0] > cur[1] for cur, nxt in zip(ordered, ordered[1:]))
        gaps_ws = all(not line[cur[1]:nxt[0] - 1].strip() for cur, nxt in zip(ordered, ordered[1:]))
        edges_ws = not line[:ordered[0][0] - 1].strip() and not line[ordered[-1][1]:].strip()
        _r["verses_with_spans"] += 1
        _r["spans_disjoint"] += disjoint
        _r["gaps_are_whitespace"] += gaps_ws
        _r["edges_are_whitespace"] += edges_ws
        _r["partitions_the_line"] += disjoint and gaps_ws and edges_ws
    if not ok:
        segment_failures.append((corpus_of_verse[_k], _k, target, built))
        _r["differ"] += 1
        _r["reordered"] += sorted(built) == sorted(target)
        _r["segments_shorter"] += len(built) < len(target)
        _r["segments_longer"] += len(built) > len(target)
        _r["text_differs_from_its_span"] += any(
            F.start_idx.v(s) is not None
            and line[F.start_idx.v(s) - 1:F.end_idx.v(s)] != (F.segment.v(s) or "")
            for s in segs)
        _r["spans_still_cover_the_line"] += letters(
            "".join(line[a - 1:b] for a, b in sorted(spans))) == target
        _r["segment_without_a_span"] += any(F.start_idx.v(s) is None for s in segs)

table("TEST 4  the segments of a verse spell the verse line", test4,
      ["verses", "segments", "repeated", "segments_spell_line", "utf8_spell_line"],
      pct=[("%", "segments_spell_line", "verses"), ("% utf8", "utf8_spell_line", "verses")])
print()
table("TEST 4  by the morphology under the segment", test4,
      ["segments", "with_morphology", "without_morphology", "analysed_verses",
       "analysed_spell_line", "rest_verses", "rest_spell_line"],
      pct=[("% segments analysed", "with_morphology", "segments"),
           ("% of analysed verses", "analysed_spell_line", "analysed_verses"),
           ("% of the rest", "rest_spell_line", "rest_verses")])
print()
table("TEST 4  the composition of the failures", test4,
      ["differ", "reordered", "segments_shorter", "segments_longer",
       "text_differs_from_its_span", "spans_still_cover_the_line", "segment_without_a_span"])
print()
table("TEST 4  how much of each line the segments emit", test4,
      ["line_letters", "segment_letters"],
      pct=[("% of the letters", "segment_letters", "line_letters")])
results["4 segments compose the verse line"] = (sum(test4[c]["verses"] for c in CORPORA),
                                                sum(test4[c]["segments_spell_line"]
                                                    for c in CORPORA))

In [ ]:
# The same segments, now read as geometry through the two tables the offsets support. `% with a
# span` gives the population that the second percentage is taken over. A segment whose slice
# returns its own text is one whose characters all belong to words that the segment holds, the
# same containment property that test 2 measures from the other side. A verse that partitions is
# one whose segments tile its line with nothing but whitespace between them.

table("TEST 4  the segment spans, resolved against the line", test4,
      ["segments", "segment_offset", "segment_no_offset", "segment_slice_is_segment"],
      pct=[("% with a span", "segment_offset", "segments"),
           ("% slice is the segment", "segment_slice_is_segment", "segment_offset")])
print()
table("TEST 4  the segment spans as a partition of the line", test4,
      ["verses", "verses_with_spans", "spans_disjoint", "gaps_are_whitespace",
       "edges_are_whitespace", "partitions_the_line"],
      pct=[("% partitioning", "partitions_the_line", "verses_with_spans")])

In [ ]:
# TEST 5. Reconstruction of the verse line from the word (slot) tier.
#
# Method: the slot type is `word`, so the reconstruction uses the text API applied to the verse.
# The format `text-trans-full` is defined as `{voc}{trailer}`, which concatenates each word with
# the material that follows it, and the result is compared to the `line` feature of the verse. The
# comparison is made on the whole string after trimming the trailing space that the final word
# contributes, so it is exact at the level of the character and includes every space.
# `letters_only` repeats the comparison under the letters class, and `utf8_spell_line` repeats the
# exact comparison in Aramaic square script, using `text-orig-full` against `line_utf8`.
#
# What it measures: whether the published slot sequence of a verse matches the published verse
# line. This is the end-to-end statement about the dataset, and the one a reader consuming the
# corpus through T.text experiences directly.
#
# Interpretation: the second table restricts the population to the verses in which every word
# carries morphological analysis, separating the verses the decomposition can be evaluated
# against from the verses where the source material supplies no analysis. The column
# `words_with_morphology` gives the same quantity at the level of the word.

test5 = defaultdict(Counter)
line_failures = []
for _k, _vns in verse_nodes.items():
    _r = test5[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    line_utf8 = max({F.line_utf8.v(v) or "" for v in _vns})
    ws = [w for v in _vns for w in L.d(v, otype="word")]
    built = "".join((F.voc.v(w) or "") + (F.trailer.v(w) or "") for w in ws)
    built_utf8 = "".join((F.voc_utf8.v(w) or "") + (F.trailer.v(w) or "") for w in ws)
    analysed = all(F.source.v(w) == "word" for w in ws)
    ok = built.strip() == line
    _r["verses"] += 1
    _r["words"] += len(ws)
    _r["words_with_morphology"] += sum(1 for w in ws if F.source.v(w) == "word")
    _r["words_spell_line"] += ok
    _r["letters_only"] += letters(built) == letters(line)
    _r["utf8_spell_line"] += built_utf8.strip() == line_utf8
    _r["every_word_analysed"] += analysed
    _r["analysed_and_spelled"] += analysed and ok
    if not ok:
        line_failures.append((corpus_of_verse[_k], _k, line, built.strip()))

table("TEST 5  the words of a verse spell the verse line, character for character", test5,
      ["verses", "words_spell_line", "letters_only", "utf8_spell_line"],
      pct=[("% exact", "words_spell_line", "verses"), ("% letters", "letters_only", "verses")])
print()
table("TEST 5  restricted to the verses in which every word carries morphology", test5,
      ["verses", "words", "words_with_morphology", "every_word_analysed", "analysed_and_spelled"],
      pct=[("% words", "words_with_morphology", "words"),
           ("% of those verses", "analysed_and_spelled", "every_word_analysed")])
results["5 words compose the verse line"] = (sum(test5[c]["verses"] for c in CORPORA),
                                             sum(test5[c]["words_spell_line"] for c in CORPORA))

In [ ]:
# Repeats test 5 through T.text and classifies its failures, then shows the point at which two of
# them first diverge.
#
# The repetition through the text API checks the hand-written concatenation above. It covers the
# verses that occupy a single verse node, because T.text takes one node at a time.
#
# The classification identifies which tier is responsible for each failure. A verse with nothing
# published to spell it holds only words whose `source` is `none`, so the manuscript records a
# line that the source material never analysed and never transcribed at the word tier. A verse in
# which some word carries no morphological analysis is short by that word. A verse in which every
# word is analysed and the reconstruction still differs represents the residue attributable to the
# alignment.

_agree = _differ = 0
for _k, _vns in verse_nodes.items():
    if len(_vns) == 1:
        if T.text(_vns[0], fmt="text-trans-full").strip() == (F.line.v(_vns[0]) or ""):
            _agree += 1
        else:
            _differ += 1
print(f"T.text(verse, fmt='text-trans-full') == line:  {_agree:,} agree, {_differ:,} differ\n")

kinds = Counter()
residue = []
for _c, _k, _line, _built in line_failures:
    ws = [w for v in verse_nodes[_k] for w in L.d(v, otype="word")]
    if not _built:
        kinds["nothing published to spell it with"] += 1
    elif any(F.source.v(w) != "word" for w in ws):
        kinds["some word carries no morphology"] += 1
    else:
        kinds["every word analysed, residue of the alignment"] += 1
        residue.append((_c, _k, _line, _built))
for kind, n in kinds.most_common():
    print(f"  {n:>5,}  {kind}")


def first_divergence(a, b):
    i = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    return i, a[max(0, i - 28):i + 28], b[max(0, i - 28):i + 28]


print()
for _c, _k, _line, _built in residue[:2]:
    at, la, lb = first_divergence(_line, _built)
    print(f"{_c} {_k[0]} uid {_k[1]}, first difference at character {at}")
    print(f"  line : ...{la}...")
    print(f"  words: ...{lb}...")

In [ ]:
# The word tier's offsets, reported here as coverage across all words, alongside the resolution
# rate for the words that carry one.
#
# A word's `start_idx` and `end_idx` are populated only where the line, sliced at those offsets,
# already returns the word's own text. Because of this, "the offsets resolve against the line" is
# true of every word that carries an offset by construction, and quoting that figure on its own
# would be quoting a tautology: 100% of a population defined as the part that passes. The number
# that carries information is how many words carry an offset at all, so both figures are printed
# together here, and the second should always be read alongside the first.
#
# The resolution rate is still worth checking, because the offset passes through several stages
# between where it is first computed and where it appears in this dataset. A fault introduced at
# any of those stages would show up here as a slice that no longer returns its word.
#
# `part` explains why a word may legitimately carry no offset. A word cut by an editorial boundary
# publishes one node per side, each holding only its own characters, so the population being
# measured is made up of word pieces, each smaller than the whole source word.

offsets = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = offsets[corpus_of_verse[_k]]
    line = max({F.line.v(v) or "" for v in _vns})
    for _w in (w for v in _vns for w in L.d(v, otype="word")):
        voc = (F.voc.v(_w) or "").strip()
        _r["words"] += 1
        _r["with_surface"] += bool(voc)
        _r["split_pieces"] += (F.part.v(_w) or 1) > 1
        a, b = F.start_idx.v(_w), F.end_idx.v(_w)
        if a is None:
            _r["no_offset"] += 1
            continue
        _r["with_offset"] += 1
        _r["slice_is_word"] += line[a - 1:b] == voc

table("the word tier's offsets, as coverage", offsets,
      ["words", "with_surface", "split_pieces", "with_offset", "no_offset", "slice_is_word"],
      pct=[("% carrying an offset", "with_offset", "words"),
           ("% of those resolving", "slice_is_word", "with_offset")])
results["word offsets carried by the word tier"] = (
    sum(offsets[c]["words"] for c in CORPORA),
    sum(offsets[c]["with_offset"] for c in CORPORA))
results["word offsets that resolve, of those carried"] = (
    sum(offsets[c]["with_offset"] for c in CORPORA),
    sum(offsets[c]["slice_is_word"] for c in CORPORA))

In [ ]:
# TEST 6. Agreement of the stored ordinals with the publication order.
#
# Method: within each verse, read the segments and the word groups in node order, the order the
# dataset publishes, and count the adjacent pairs whose stored ordinal decreases. For segments the
# ordinals are `rank` and `start_idx`. For word groups the ordinal is the pair (`rank`, `reading`),
# because groups competing for one position share a rank and are ordered among themselves by
# reading. The final column tests whether the ranks of the word groups of a verse form a dense
# sequence starting from zero, the property that makes rank usable as a word index.
#
# What it measures: whether a consumer reading the corpus in node order and a consumer sorting by
# the stored ordinals obtain the same sequence. An inversion would allow a verse to be published in
# an order that its own features contradict, a fault the preceding tests would not detect, because
# each of them reads in node order throughout.

test6 = defaultdict(Counter)
for _k, _vns in verse_nodes.items():
    _r = test6[corpus_of_verse[_k]]
    _r["verses"] += 1
    segs = [s for v in _vns for s in L.d(v, otype="segment")]
    ranks = [F.rank.v(s) for s in segs]
    starts = [F.start_idx.v(s) for s in segs]
    _r["segments"] += len(segs)
    _r["segment_rank_inversions"] += sum(
        1 for a, b in zip(ranks, ranks[1:]) if a is not None and b is not None and b < a)
    _r["segment_span_inversions"] += sum(
        1 for a, b in zip(starts, starts[1:]) if a is not None and b is not None and b < a)
    gs = [(F.rank.v(g), F.reading.v(g)) for v in _vns for g in L.d(v, otype="word_group")]
    _r["word_groups"] += len(gs)
    _r["group_order_inversions"] += sum(1 for a, b in zip(gs, gs[1:]) if b < a)
    slots = {rank for rank, _ in gs}
    _r["slot_rank_dense"] += sorted(slots) == list(range(len(slots)))

table("TEST 6  stored ordinals against publication order", test6,
      ["verses", "segments", "segment_rank_inversions", "segment_span_inversions",
       "word_groups", "group_order_inversions", "slot_rank_dense"],
      pct=[("% dense", "slot_rank_dense", "verses")])
results["6 ordinals agree with node order"] = (
    sum(test6[c]["verses"] for c in CORPORA), sum(test6[c]["slot_rank_dense"] for c in CORPORA))

In [ ]:
# Collects the six tests into one table. The population column states what was counted, which
# differs by test: groups for test 1, segments for test 2, and verses for tests 3 through 6. This
# is the summary to consult after the dataset is regenerated, because a change in any row reflects
# a change in the data or in the boundary of what the dataset publishes.

print(f"  {'test':<40} {'population':>12} {'pass':>12} {'%':>8}")
print("  " + "-" * 76)
for name, (n, ok) in results.items():
    print(f"  {name:<40} {n:>12,} {ok:>12,} {100.0 * ok / n:>8.2f}")